# Modern LLM Architecture

Explore the architectural innovations that define state-of-the-art LLMs: rotary position embeddings, efficient normalization, gated linear units, and multi-query attention. Understand the "why" behind each design choice.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/modern-llm-arch)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Colab provides PyTorch with CUDA support, so these notebooks run on a real GPU rather than Apple's MPS backend. If a code sample uses `device = torch.device('mps' ...)`, it will fall back to `cpu` on Colab unless you adapt it; replace `'mps'` with `'cuda'` for GPU execution.


In [ ]:
!pip install torch numpy

---

## Lesson examples (optional)

These are the runnable code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.


### Lesson example: RoPE: Rotary Position Embeddings


In [ ]:
import torch
import math

def apply_rope(q, k, positions, dim, theta_base=10000.0):
    """
    Apply RoPE to query and key vectors.

    Args:
        q: (batch, seq_len, n_heads, head_dim)
        k: (batch, seq_len, n_heads, head_dim)
        positions: (seq_len,), position indices
        dim: head_dim
        theta_base: base for frequency scaling

    Returns:
        q_rotated, k_rotated (same shape)
    """
    # Compute frequencies: theta_i = theta_base^(-2i/dim)
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, dim, 2).float() / dim))

    # Compute rotation angles: pos * theta_i
    # positions: (seq_len,)
    # inv_freq: (dim // 2,)
    t = torch.einsum('s, d -> s d', positions.float(), inv_freq)  # (seq_len, dim//2)

    # Duplicate for interleaved dimensions
    freqs = torch.cat([t, t], dim=-1)  # (seq_len, dim)

    # Compute cos and sin
    cos_freq = torch.cos(freqs)  # (seq_len, dim)
    sin_freq = torch.sin(freqs)  # (seq_len, dim)

    # Apply rotation: [x_0, x_1, x_2, x_3, ...] ->
    #                 [x_0*cos - x_1*sin, x_1*cos + x_0*sin, ...]
    def rotate_half(x):
        x1, x2 = x[..., :dim//2], x[..., dim//2:]
        return torch.cat([-x2, x1], dim=-1)

    q_rot = q * cos_freq.unsqueeze(0).unsqueeze(2) + rotate_half(q) * sin_freq.unsqueeze(0).unsqueeze(2)
    k_rot = k * cos_freq.unsqueeze(0).unsqueeze(2) + rotate_half(k) * sin_freq.unsqueeze(0).unsqueeze(2)

    return q_rot, k_rot

# Example
batch, seq_len, n_heads, head_dim = 2, 8, 4, 64
q = torch.randn(batch, seq_len, n_heads, head_dim)
k = torch.randn(batch, seq_len, n_heads, head_dim)
positions = torch.arange(seq_len)

q_rope, k_rope = apply_rope(q, k, positions, head_dim)
print(f"RoPE applied: q shape {q_rope.shape}, k shape {k_rope.shape}")

# Verify relative position: the dot product depends only on m-n
m, n = 3, 1
rel_pos = m - n
# The angle difference should be rel_pos * theta_i
print(f"Relative position: {rel_pos}, this is encoded in the rotation")

# YaRN extrapolation (scaled RoPE)
def apply_rope_yarn(q, k, positions, dim, max_position=4096, scaling_factor=1.0):
    """RoPE with YaRN-style frequency scaling for longer sequences."""
    inv_freq = 1.0 / (10000.0 ** (torch.arange(0, dim, 2).float() / dim))
    # Scale frequencies for longer sequences
    inv_freq = inv_freq / scaling_factor

    t = torch.einsum('s, d -> s d', positions.float(), inv_freq)
    freqs = torch.cat([t, t], dim=-1)
    cos_freq = torch.cos(freqs)
    sin_freq = torch.sin(freqs)

    def rotate_half(x):
        x1, x2 = x[..., :dim//2], x[..., dim//2:]
        return torch.cat([-x2, x1], dim=-1)

    q_rot = q * cos_freq.unsqueeze(0).unsqueeze(2) + rotate_half(q) * sin_freq.unsqueeze(0).unsqueeze(2)
    k_rot = k * cos_freq.unsqueeze(0).unsqueeze(2) + rotate_half(k) * sin_freq.unsqueeze(0).unsqueeze(2)

    return q_rot, k_rot

# Test: can now handle longer sequences
positions_long = torch.arange(8192)
q_long = torch.randn(1, 8192, 1, 64)
k_long = torch.randn(1, 8192, 1, 64)
q_rope_long, k_rope_long = apply_rope_yarn(q_long, k_long, positions_long, 64, scaling_factor=2.0)
print(f"YaRN extended to 8k context: {q_rope_long.shape}")


### Lesson example: RMSNorm


In [ ]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    """RMS Normalization."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # RMS = sqrt(mean(x^2))
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        # Normalize and scale
        return x / rms * self.scale

# Compare RMSNorm vs LayerNorm
dim = 64
x = torch.randn(4, 8, dim)

rms_norm = RMSNorm(dim)
layer_norm = nn.LayerNorm(dim)

x_rms = rms_norm(x)
x_ln = layer_norm(x)

print(f"RMSNorm output stats: mean={x_rms.mean():.6f}, std={x_rms.std():.4f}")
print(f"LayerNorm output stats: mean={x_ln.mean():.6f}, std={x_ln.std():.4f}")

# Both stabilize activations, RMSNorm is slightly simpler
# In practice, RMSNorm is often numerically equivalent for deep networks

# Modern LLM block with RMSNorm
class ModernBlock(nn.Module):
    def __init__(self, d_model, d_ff, n_heads):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.ln2 = RMSNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x), self.ln1(x), self.ln1(x))[0]
        x = x + self.ffn(self.ln2(x))
        return x

block = ModernBlock(512, 2048, 8)
x = torch.randn(2, 10, 512)
y = block(x)
print(f"Block output shape: {y.shape}")


### Lesson example: SwiGLU FFN


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SwiGLU(nn.Module):
    """SwiGLU Feed-Forward Network."""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = int(d_model * 8 / 3)  # 8/3 rule
        # Project to 2*d_ff to split into gate and up
        self.w_gate = nn.Linear(d_model, d_ff)
        self.w_up = nn.Linear(d_model, d_ff)
        self.w_down = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # Gate and up projections
        gate = self.w_gate(x)  # (batch, seq, d_ff)
        up = self.w_up(x)      # (batch, seq, d_ff)

        # Apply SiLU to gate and multiply
        x = F.silu(gate) * up

        # Project back down
        x = self.w_down(x)
        return x

# Compare traditional FFN vs SwiGLU
d_model = 512
d_ff_traditional = 4 * d_model  # 2048

swiglu = SwiGLU(d_model, d_ff=int(d_model * 8 / 3))  # ~1365
traditional = nn.Sequential(
    nn.Linear(d_model, d_ff_traditional),
    nn.ReLU(),
    nn.Linear(d_ff_traditional, d_model)
)

x = torch.randn(2, 10, d_model)

# Count parameters
swiglu_params = sum(p.numel() for p in swiglu.parameters())
traditional_params = sum(p.numel() for p in traditional.parameters())
print(f"SwiGLU params: {swiglu_params}, Traditional params: {traditional_params}")
print(f"SwiGLU uses ~{swiglu_params / traditional_params:.1%} of traditional FFN params")

# Output
y_swiglu = swiglu(x)
y_trad = traditional(x)
print(f"Both output shape: {y_swiglu.shape}")

# Modern FFN with gating
class GLU_FFN(nn.Module):
    """Generic GLU variant (gate, up, down)."""
    def __init__(self, d_model, d_ff=None, activation='swiglu'):
        super().__init__()
        if d_ff is None:
            d_ff = int(d_model * 8 / 3)
        self.w_gate = nn.Linear(d_model, d_ff)
        self.w_up = nn.Linear(d_model, d_ff)
        self.w_down = nn.Linear(d_ff, d_model)
        self.activation = activation

    def forward(self, x):
        gate = self.w_gate(x)
        up = self.w_up(x)
        if self.activation == 'swiglu':
            x = F.silu(gate) * up
        elif self.activation == 'gelu':
            x = F.gelu(gate) * up
        return self.w_down(x)

glu_ffn = GLU_FFN(d_model, activation='gelu')
y = glu_ffn(x)
print(f"GLU FFN output: {y.shape}")


### Lesson example: GQA: Grouped Query Attention


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def grouped_query_attention(q, k, v, n_kv_heads, mask=None):
    """
    Grouped Query Attention (GQA).

    Args:
        q: (batch, seq_len, n_q_heads, head_dim)
        k: (batch, seq_len, n_kv_heads, head_dim)
        v: (batch, seq_len, n_kv_heads, head_dim)
        n_kv_heads: number of KV heads
        mask: (seq_len, seq_len) causal mask (optional)

    Returns:
        output: (batch, seq_len, n_q_heads, head_dim)
    """
    batch, seq_len, n_q_heads, head_dim = q.shape
    n_kv_heads = k.shape[2]
    num_q_per_kv = n_q_heads // n_kv_heads

    # Expand K/V to match Q groups
    k_expanded = k.repeat_interleave(num_q_per_kv, dim=2)  # (batch, seq_len, n_q_heads, head_dim)
    v_expanded = v.repeat_interleave(num_q_per_kv, dim=2)  # (batch, seq_len, n_q_heads, head_dim)

    # Compute attention scores: (batch, n_q_heads, seq_len, seq_len)
    scores = torch.matmul(q.transpose(1, 2), k_expanded.transpose(1, 2).transpose(-2, -1))
    scores = scores / (head_dim ** 0.5)

    # Apply causal mask
    if mask is not None:
        scores = scores + mask.unsqueeze(0).unsqueeze(0) * -1e9

    # Softmax and dropout
    attn_weights = F.softmax(scores, dim=-1)

    # Apply attention to values: (batch, n_q_heads, seq_len, head_dim)
    output = torch.matmul(attn_weights, v_expanded.transpose(1, 2))

    # Reshape back: (batch, seq_len, n_q_heads, head_dim)
    output = output.transpose(1, 2)

    return output

# Test GQA vs MHA
batch, seq_len, head_dim = 2, 8, 64
n_q_heads = 32
n_kv_heads = 4  # GQA: 8x reduction in KV heads

q = torch.randn(batch, seq_len, n_q_heads, head_dim)
k = torch.randn(batch, seq_len, n_kv_heads, head_dim)
v = torch.randn(batch, seq_len, n_kv_heads, head_dim)

output = grouped_query_attention(q, k, v, n_kv_heads)
print(f"GQA output shape: {output.shape}")
print(f"KV cache size (GQA): {k.numel() + v.numel()} vs (MHA): {batch * seq_len * n_q_heads * head_dim * 2}")

# Module version
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads, head_dim):
        super().__init__()
        assert n_q_heads % n_kv_heads == 0
        self.d_model = d_model
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim
        self.num_q_per_kv = n_q_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_q_heads * head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim)
        self.out_proj = nn.Linear(n_q_heads * head_dim, d_model)

    def forward(self, x, mask=None):
        batch, seq_len, _ = x.shape

        # Project to Q, K, V
        q = self.q_proj(x).view(batch, seq_len, self.n_q_heads, self.head_dim)
        k = self.k_proj(x).view(batch, seq_len, self.n_kv_heads, self.head_dim)
        v = self.v_proj(x).view(batch, seq_len, self.n_kv_heads, self.head_dim)

        # GQA
        output = grouped_query_attention(q, k, v, self.n_kv_heads, mask)

        # Project out
        output = output.view(batch, seq_len, -1)
        output = self.out_proj(output)

        return output

gqa = GroupedQueryAttention(d_model=512, n_q_heads=32, n_kv_heads=4, head_dim=64)
x = torch.randn(2, 10, 512)
output = gqa(x)
print(f"GQA module output: {output.shape}")


---

## Exercise: Build a Modern Llama-style Block


Implement a Transformer block using modern techniques: RoPE (or simple absolute positions), RMSNorm, SwiGLU FFN, and GQA. Ensure the block computes attention correctly and integrates all components. Test with example inputs.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.scale

class SwiGLU(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        d_ff = int(d_model * 8 / 3)
        self.w_gate = nn.Linear(d_model, d_ff)
        self.w_up = nn.Linear(d_model, d_ff)
        self.w_down = nn.Linear(d_ff, d_model)

    def forward(self, x):
        gate = self.w_gate(x)
        up = self.w_up(x)
        x = F.silu(gate) * up
        return self.w_down(x)

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads):
        super().__init__()
        assert n_q_heads % n_kv_heads == 0
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_q_heads

        self.q_proj = nn.Linear(d_model, n_q_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_q_heads * self.head_dim, d_model)

    def forward(self, x, mask=None):
        # TODO: Implement GQA forward pass
        # 1. Project to Q, K, V
        # 2. Expand K/V to match Q groups (repeat_interleave)
        # 3. Compute attention scores and apply mask
        # 4. Softmax and apply to values
        # 5. Project output
        pass

class ModernTransformerBlock(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads):
        super().__init__()
        # TODO: Initialize components
        # - RMSNorm for pre-norm
        # - GQA attention
        # - SwiGLU FFN
        pass

    def forward(self, x, mask=None):
        # TODO: Implement block forward pass
        # Use pre-norm: residual + attention(norm(x)), residual + ffn(norm(x))
        pass

if __name__ == '__main__':
    # Test
    d_model = 512
    n_q_heads = 32
    n_kv_heads = 4
    seq_len = 10
    batch = 2

    block = ModernTransformerBlock(d_model, n_q_heads, n_kv_heads)
    x = torch.randn(batch, seq_len, d_model)

    output = block(x)
    print(f"Output shape: {output.shape}")
    assert output.shape == x.shape, "Output shape mismatch"
    print("✓ Block test passed!")

---

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/modern-llm-arch) for the solution and discussion.
